In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q datasets

# 라이브러리

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# Torch 모델 구조
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT Model
# Fine Tuning
from transformers import AutoModel, AutoTokenizer
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput


# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [ ]:
class ReviewPairDataset(Dataset):
    def __init__(self, user_texts, item_texts, user_ids, item_ids, ratings, tokenizer, max_length=512):
        self.user_texts = user_texts
        self.item_texts = item_texts
        self.user_ids = user_ids
        self.item_ids = item_ids
        self.ratings = ratings
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.user_texts)

    def __getitem__(self, idx):
        user_enc = self.tokenizer(self.user_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        item_enc = self.tokenizer(self.item_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        user_id = torch.tensor(self.user_ids[idx], dtype=torch.long)
        item_id = torch.tensor(self.item_ids[idx], dtype=torch.long)
        rating = torch.tensor(float(self.ratings[idx]), dtype=torch.float)

        return {
            "input_ids_user": user_enc["input_ids"].squeeze(0),
            "attention_mask_user": user_enc["attention_mask"].squeeze(0),
            "input_ids_item": item_enc["input_ids"].squeeze(0),
            "attention_mask_item": item_enc["attention_mask"].squeeze(0),
            "labels": rating,
            "user_id": user_id,
            "item_id": item_id
        }

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions  # 그대로 사용!

    # 지표 계산
    mae = mean_absolute_error(labels, preds)
    mse = mean_squared_error(labels, preds)
    rmse = np.sqrt(mse)

    epsilon = 1e-8
    mape = np.mean(np.abs((labels - preds) / (labels + epsilon))) * 100

    return {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "mape": mape
    }

# 데이터셋 로딩

In [ ]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_aspect_final.pkl')
print(df.shape)
df.head()

(23491, 11)


,user_id,asin,parent_asin,text,rating,title_y,small_image_url,medium_image_url,large_image_url,LLaMA_Output,LLaVA_Output
0,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0BJKYJR3L,B0BJKYJR3L,"Nice privacy shield. Good color, fits well, et...",3,Jephuwneh 2 Pack Gradient Tempered Glass for i...,https://m.media-amazon.com/images/I/515oPLx13k...,https://m.media-amazon.com/images/I/515oPLx13k...,https://m.media-amazon.com/images/I/515oPLx13k...,"Based on this user review, it appears that the...",The image shows a cellphone with a screen prot...
1,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0BHTZQHQW,B0BHTZQHQW,"This thing is great. It's not too big, has a d...",5,"NEOCOR Magnetic Wireless Power Bank, 10,000mAh...",https://m.media-amazon.com/images/I/71K8BDxzsA...,https://m.media-amazon.com/images/I/71K8BDxzsA...,https://m.media-amazon.com/images/I/71K8BDxzsA...,"Based on the user review, here is a summary of...",The image shows a wireless power bank with a m...
2,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0B9N4VRB9,B0BHSSCXB2,This is a much better way to mount my iPhone (...,5,Lamicall Gooseneck Phone Holder Bed - [Newest ...,https://m.media-amazon.com/images/I/71b+Sx-Jq7...,https://m.media-amazon.com/images/I/71b+Sx-Jq7...,https://m.media-amazon.com/images/I/71b+Sx-Jq7...,"Based on the user review, here is a summary of...",The image shows a cellphone holder with a flex...
3,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0B93H178F,B0B93H178F,"This is a fantastic charger, especially if you...",5,"Shargeek USB C Charger 140W Wall Charger, Mult...",https://m.media-amazon.com/images/I/61R6UXrAKT...,https://m.media-amazon.com/images/I/61R6UXrAKT...,https://m.media-amazon.com/images/I/61R6UXrAKT...,"Based on the user review, here is a summary of...",The image shows a USB C charger with a power r...
4,AFERCDY2EFJKT7QUQ75GISNHTFOQ,B0BPXZWCBG,B0BPY47SY6,There never seems to be enough charging cords ...,4,"Charging Station for Multiple Devices,60W 12A ...",https://m.media-amazon.com/images/I/51fqzioGKe...,https://m.media-amazon.com/images/I/51fqzioGKe...,https://m.media-amazon.com/images/I/51fqzioGKe...,"Based on user reviews, it appears that users v...",The image shows a product packaging for a USB ...


# Prepare Input

In [ ]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

# fit-transform
df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 실험 데이터의 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

Num_User = n_users + 1
Num_Item = n_items + 1

In [ ]:
# train, valid, test으로 데이터셋 분리
train, test = train_test_split(df, test_size=0.2, random_state=42)
train, valid = train_test_split(train, test_size=0.125, random_state=42)

train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [ ]:
train.shape, valid.shape, test.shape

((16443, 11), (2349, 11), (4699, 11))

In [ ]:
n_users, n_items

(4149, 4975)

In [ ]:
# sparsity = 1 - (len(df) / (n_users * n_items))
# print(sparsity)

In [ ]:
# train, test, valid 변수 선언
train_user_id = train['user_id'].to_numpy()
train_item_id = train['asin'].to_numpy()
train_summary = train['LLaMA_Output']
train_image_description = train['LLaVA_Output']
train_rating = train['rating'].to_numpy()

valid_user_id = valid['user_id'].to_numpy()
valid_item_id = valid['asin'].to_numpy()
valid_summary = valid['LLaMA_Output']
valid_image_description = valid['LLaVA_Output']
valid_rating = valid['rating'].to_numpy()

test_user_id = test['user_id'].to_numpy()
test_item_id = test['asin'].to_numpy()
test_summary = test['LLaMA_Output']
test_image_description = test['LLaVA_Output']
test_rating = test['rating'].to_numpy()

# Load Model

In [ ]:
# BERT 모델 Loading
# 해당 모델은 Layer 6개인 모델

# 사용자 텍스트 처리 모델
BERT_review = AutoModel.from_pretrained("bert-base-uncased")

# 아이템 텍스트 처리 모델
BERT_image = AutoModel.from_pretrained("bert-base-uncased")

# tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Two Tower BERT

```python
MAS_Rec(
  user_id: Embedding(num_users, 64)
  item_id: Embedding(num_items, 64)
  user_model: BertModel
  item_model: BertModel
  cls_dense: MLP(768→512→256)
  concat: Concat([user_emb, item_emb])
  fc_final: MLP(640→256→1)
)

```

In [ ]:
def make_mlp(sizes, activation=nn.ReLU):
    """
    중간 feature extractor용 MLP
    모든 Linear 뒤에 activation 함수가 붙음 (마지막 레이어 포함)
    """
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        layers.append(activation())
    return nn.Sequential(*layers)

In [ ]:
def make_last_mlp(sizes, activation=nn.ReLU):
    """
    최종 예측 MLP
    마지막 Linear 뒤에는 activation이 없음 (즉, 마지막은 Linear-only)
    """
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        if i < len(sizes) - 2:
            layers.append(activation())
    return nn.Sequential(*layers)

In [ ]:
class MAS_Rec(nn.Module):
    def __init__(self, n_users, n_items, user_model, item_model):
        super().__init__()

        # Interaction IDs
        self.user_id = nn.Embedding(n_users, 64)
        self.item_id = nn.Embedding(n_items, 64)

        # Two Tower BERT
        self.user_model = user_model
        self.item_model = item_model

        # [CLS] Dense
        self.cls_dense = make_mlp([768, 512, 256])

        # GMU gate
        self.gmu_gate = nn.Linear(640, 320)

        # Last Dense
        # GMU output dimension is 320
        self.fc_final = make_last_mlp([320, 128, 1])

    def forward(self, user_id, item_id, input_ids_user, attention_mask_user, input_ids_item, attention_mask_item, labels=None):

        # Interaction IDs
        user_emb = self.user_id(user_id)    # [batch, 64]
        item_emb = self.item_id(item_id)    # [batch, 64]

        # Two Tower BERT
        # 여기에서 input_ids는 token_id(token index)
        user_cls = self.user_model(input_ids=input_ids_user, attention_mask=attention_mask_user).last_hidden_state[:, 0, :]
        item_cls = self.item_model(input_ids=input_ids_item, attention_mask=attention_mask_item).last_hidden_state[:, 0, :]

        # [CLS] Dense
        user_cls = self.cls_dense(user_cls)   # [batch, 256]
        item_cls = self.cls_dense(item_cls)   # [batch, 256]

        # User / Item Feature
        user_feat = torch.cat([user_emb, user_cls], dim=-1)   # [batch, 320]
        item_feat = torch.cat([item_emb, item_cls], dim=-1)   # [batch, 320]

        # =========================
        # GMU Fusion
        # z = sigmoid(Wz [user_feat ; item_feat] + bz)
        # h = z ⊙ user_feat + (1-z) ⊙ item_feat
        # =========================
        gate_input = torch.cat([user_feat, item_feat], dim=-1)   # [batch, 640]
        z = torch.sigmoid(self.gmu_gate(gate_input))              # [batch, 320]

        final_feat = z * user_feat + (1.0 - z) * item_feat       # [batch, 320]

        # Last Dense Layer
        logits = self.fc_final(final_feat).squeeze(1)

        if labels is not None:
            loss = nn.MSELoss()(logits, labels)
            return {"loss": loss, "logits": logits}

        return {"logits": logits}

# Fine Tuning

In [ ]:
# 모델 생성
model = MAS_Rec(Num_User, Num_Item, user_model=BERT_review, item_model=BERT_image)

In [ ]:
for name, module in model.named_children():
    print(name, module)

user_id Embedding(4150, 64)
item_id Embedding(4976, 64)
user_model BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [ ]:
# 실제 훈련용
train_dataset = ReviewPairDataset(train_summary, train_image_description, train_user_id, train_item_id, train_rating, tokenizer, max_length=512)
valid_dataset = ReviewPairDataset(valid_summary, valid_image_description, valid_user_id, valid_item_id, valid_rating, tokenizer, max_length=512)
test_dataset = ReviewPairDataset(test_summary, test_image_description, test_user_id, test_item_id, test_rating, tokenizer, max_length=512)

In [ ]:
# 하이퍼파라미터
num_training_samples = 46702 # 46702 # 16443

# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir="./logs",
    metric_for_best_model="mse",  # ✅ compute_metrics에 꼭 있어야 함
    greater_is_better=False,
    fp16=torch.cuda.is_available(),  # ✅ 안전함
)

# ✅ Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,  # 실험용으로 train=eval
    compute_metrics=compute_metrics,
)

# ✅ 학습 시작
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse,Mape
1,4.435705,0.333603,0.355418,0.333603,0.577584,10.895517
2,0.228998,0.335647,0.353138,0.335647,0.579351,11.130292
3,0.204218,0.269799,0.290525,0.269799,0.519422,9.132540
4,0.181842,0.275007,0.301712,0.275007,0.524412,9.302461
5,0.168305,0.291227,0.309118,0.291227,0.539655,9.613396


TrainOutput(global_step=2570, training_loss=1.0438134991241337, metrics={'train_runtime': 871.1114, 'train_samples_per_second': 94.379, 'train_steps_per_second': 2.95, 'total_flos': 0.0, 'train_loss': 1.0438134991241337, 'epoch': 5.0})

# Train

In [ ]:
# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=100,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=10,
    metric_for_best_model="mse",  # ✅ compute_metrics에 꼭 있어야 함
    greater_is_better=False,
    fp16=torch.cuda.is_available(),  # ✅ 안전함
)

# ✅ Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,  # 실험용으로 train=eval
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]  # ✅ 여기에 patience 설정
)

# ✅ 학습 시작
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse,Mape
1,0.105497,0.317879,0.328349,0.317879,0.563808,10.163043
2,0.144831,0.324104,0.344978,0.324104,0.569301,10.411201
3,0.132312,0.282342,0.296685,0.282342,0.531359,9.347894
4,0.147123,0.305207,0.349937,0.305207,0.552455,10.081828
5,0.097838,0.334384,0.353867,0.334384,0.578260,10.275290
6,0.091960,0.348233,0.355911,0.348233,0.590112,10.760704
7,0.062039,0.343730,0.346193,0.343730,0.586285,10.404643
8,0.051868,0.334669,0.367596,0.334669,0.578505,10.775387


TrainOutput(global_step=4112, training_loss=0.11355135502141506, metrics={'train_runtime': 1397.1723, 'train_samples_per_second': 1176.877, 'train_steps_per_second': 36.789, 'total_flos': 0.0, 'train_loss': 0.11355135502141506, 'epoch': 8.0})

In [ ]:
# 평가
results = trainer.evaluate()

# 평가 결과 출력
print("Evaluation Results:")
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

Evaluation Results:
eval_loss: 0.2823
eval_mae: 0.2967
eval_mse: 0.2823
eval_rmse: 0.5314
eval_mape: 9.3479
eval_runtime: 9.9562
eval_samples_per_second: 235.9340
eval_steps_per_second: 3.7160
epoch: 8.0000


In [ ]:
# pd.set_option('display.max_colwidth', None)

# error_sample = df.loc[df['LLaMA_Output'].str.contains('Input')==True, ['title_y', 'text', 'LLaMA_Output']].tail(10)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel

# 데이터 입력
data = {
    "Dot": {
        "MAE": [0.5459,0.5327,0.5296,0.5366,0.5499],
        "MSE": [0.5127,0.5003,0.4891,0.5005,0.5228],
        "RMSE": [0.7160,0.7073,0.6993,0.7075,0.7231],
        "MAPE": [0.1375,0.1384,0.1369,0.1390,0.1383]
    },
    "Elementwise": {
        "MAE": [0.3689,0.3405,0.3355,0.3829,0.3466],
        "MSE": [0.2918,0.2812,0.2746,0.2977,0.2902],
        "RMSE": [0.5402,0.5303,0.5240,0.5456,0.5387],
        "MAPE": [0.1008,0.0956,0.0945,0.1033,0.0950]
    },
    "Concat": {
        "MAE": [0.2983,0.3143,0.3155,0.3051,0.3309],
        "MSE": [0.2494,0.2723,0.2702,0.2550,0.2649],
        "RMSE": [0.4994,0.5218,0.5198,0.5049,0.5147],
        "MAPE": [0.0853,0.0884,0.0888,0.0869,0.0916]
    },
    "GMU": {
        "MAE": [0.3167,0.2940,0.2968,0.3071,0.2927],
        "MSE": [0.2646,0.2485,0.2485,0.2535,0.2487],
        "RMSE": [0.5144,0.4985,0.4985,0.5034,0.4987],
        "MAPE": [0.0895,0.0833,0.0835,0.0883,0.0846]
    },
    "Proposed": {
        "MAE": [0.3016,0.3333,0.3160,0.3173,0.3108],
        "MSE": [0.2505,0.2656,0.2551,0.2615,0.2541],
        "RMSE": [0.5005,0.5153,0.5051,0.5114,0.5041],
        "MAPE": [0.0857,0.0925,0.0884,0.0902,0.0879]
    }
}

metrics = ["MAE", "MSE", "RMSE", "MAPE"]
baseline_models = ["Dot", "Elementwise", "Concat", "GMU"]
proposed_model = "Proposed"

# 결과 저장용 리스트
tstat_rows = []
pvalue_rows = []

for baseline in baseline_models:
    tstat_row = {"Model": f"{proposed_model} vs {baseline}"}
    pvalue_row = {"Model": f"{proposed_model} vs {baseline}"}

    for metric in metrics:
        proposed_scores = np.array(data[proposed_model][metric])
        baseline_scores = np.array(data[baseline][metric])

        t_stat, p_val = ttest_rel(baseline_scores, proposed_scores)

        tstat_row[metric] = round(t_stat, 4)
        pvalue_row[metric] = round(p_val, 4)

    tstat_rows.append(tstat_row)
    pvalue_rows.append(pvalue_row)

# DataFrame 생성
tstat_df = pd.DataFrame(tstat_rows)
pvalue_df = pd.DataFrame(pvalue_rows)

# 출력
print("T-statistic")
print(tstat_df.to_string(index=False))

print("\n" + "="*60 + "\n")

print("P-value")
print(pvalue_df.to_string(index=False))

T-statistic
                  Model     MAE     MSE    RMSE    MAPE
        Proposed vs Dot 26.9429 33.6539 35.4636 49.4925
Proposed vs Elementwise  3.2398  5.8291  5.8778  3.9644
     Proposed vs Concat -0.4491  1.2741  1.2559 -0.5301
        Proposed vs GMU -1.6318 -0.9003 -0.9094 -1.4651


P-value
                  Model    MAE    MSE   RMSE   MAPE
        Proposed vs Dot 0.0000 0.0000 0.0000 0.0000
Proposed vs Elementwise 0.0317 0.0043 0.0042 0.0166
     Proposed vs Concat 0.6766 0.2716 0.2775 0.6241
        Proposed vs GMU 0.1781 0.4189 0.4146 0.2168


In [ ]:
tstat_df

,Model,MAE,MSE,RMSE,MAPE
0,Proposed vs Dot,26.9429,33.6539,35.4636,49.4925
1,Proposed vs Elementwise,3.2398,5.8291,5.8778,3.9644
2,Proposed vs Concat,-0.4491,1.2741,1.2559,-0.5301
3,Proposed vs GMU,-1.6318,-0.9003,-0.9094,-1.4651


In [ ]:
pvalue_df

,Model,MAE,MSE,RMSE,MAPE
0,Proposed vs Dot,0.0000,0.0000,0.0000,0.0000
1,Proposed vs Elementwise,0.0317,0.0043,0.0042,0.0166
2,Proposed vs Concat,0.6766,0.2716,0.2775,0.6241
3,Proposed vs GMU,0.1781,0.4189,0.4146,0.2168


In [ ]:
data = {
    "Dot": {
        "MAE": [0.5277, 0.5330, 0.5516, 0.5285, 0.5346],
        "MSE": [0.5096, 0.5062, 0.5443, 0.5106, 0.5141],
        "RMSE": [0.7139, 0.7115, 0.7377, 0.7145, 0.7170],
        "MAPE": [0.1409, 0.1379, 0.1447, 0.1406, 0.1393]
    },
    "Elementwise": {
        "MAE": [0.3586, 0.3763, 0.3596, 0.3828, 0.4009],
        "MSE": [0.3126, 0.3253, 0.3198, 0.3289, 0.3422],
        "RMSE": [0.5591, 0.5704, 0.5655, 0.5735, 0.5850],
        "MAPE": [0.1044, 0.1059, 0.1058, 0.1070, 0.1120]
    },
    "Concat": {
        "MAE": [0.2987, 0.3356, 0.3113, 0.3057, 0.3010],
        "MSE": [0.2833, 0.3050, 0.2917, 0.2812, 0.2862],
        "RMSE": [0.5322, 0.5522, 0.5401, 0.5302, 0.5350],
        "MAPE": [0.0902, 0.1014, 0.0960, 0.0934, 0.0918]
    },
    "GMU": {
        "MAE": [0.2934, 0.3186, 0.2984, 0.2998, 0.2967],
        "MSE": [0.2827, 0.2860, 0.2788, 0.2816, 0.2823],
        "RMSE": [0.5317, 0.5347, 0.5280, 0.5307, 0.5314],
        "MAPE": [0.0929, 0.0965, 0.0901, 0.0904, 0.0935]
    },
    "Proposed": {
        "MAE": [0.2966, 0.3082, 0.2845, 0.3058, 0.3001],
        "MSE": [0.2817, 0.2818, 0.2703, 0.2794, 0.2768],
        "RMSE": [0.5307, 0.5308, 0.5199, 0.5286, 0.5261],
        "MAPE": [0.0909, 0.0912, 0.0884, 0.0905, 0.0906]
    }
}

metrics = ["MAE", "MSE", "RMSE", "MAPE"]
baseline_models = ["Dot", "Elementwise", "Concat", "GMU"]
proposed_model = "Proposed"

# 결과 저장용 리스트
tstat_rows = []
pvalue_rows = []

for baseline in baseline_models:
    tstat_row = {"Model": f"{proposed_model} vs {baseline}"}
    pvalue_row = {"Model": f"{proposed_model} vs {baseline}"}

    for metric in metrics:
        proposed_scores = np.array(data[proposed_model][metric])
        baseline_scores = np.array(data[baseline][metric])

        t_stat, p_val = ttest_rel(baseline_scores, proposed_scores)

        tstat_row[metric] = round(t_stat, 4)
        pvalue_row[metric] = round(p_val, 4)

    tstat_rows.append(tstat_row)
    pvalue_rows.append(pvalue_row)

# DataFrame 생성
tstat_df = pd.DataFrame(tstat_rows)
pvalue_df = pd.DataFrame(pvalue_rows)

# 출력
print("T-statistic")
print(tstat_df.to_string(index=False))

print("\n" + "="*60 + "\n")

print("P-value")
print(pvalue_df.to_string(index=False))

T-statistic
                  Model     MAE     MSE    RMSE    MAPE
        Proposed vs Dot 29.3247 26.5095 28.4392 31.3430
Proposed vs Elementwise 11.5910  8.5805  8.8136 12.3014
     Proposed vs Concat  1.7812  2.4718  2.4743  2.0908
        Proposed vs GMU  0.5745  3.2640  3.2725  2.6763


P-value
                  Model    MAE    MSE   RMSE   MAPE
        Proposed vs Dot 0.0000 0.0000 0.0000 0.0000
Proposed vs Elementwise 0.0003 0.0010 0.0009 0.0003
     Proposed vs Concat 0.1495 0.0688 0.0686 0.1047
        Proposed vs GMU 0.5964 0.0310 0.0307 0.0554


In [ ]:
tstat_df

,Model,MAE,MSE,RMSE,MAPE
0,Proposed vs Dot,29.3247,26.5095,28.4392,31.3430
1,Proposed vs Elementwise,11.5910,8.5805,8.8136,12.3014
2,Proposed vs Concat,1.7812,2.4718,2.4743,2.0908
3,Proposed vs GMU,0.5745,3.2640,3.2725,2.6763


In [ ]:
pvalue_df

,Model,MAE,MSE,RMSE,MAPE
0,Proposed vs Dot,0.0000,0.0000,0.0000,0.0000
1,Proposed vs Elementwise,0.0003,0.0010,0.0009,0.0003
2,Proposed vs Concat,0.1495,0.0688,0.0686,0.1047
3,Proposed vs GMU,0.5964,0.0310,0.0307,0.0554
